In [1]:
# Step 1: Import our essential Python libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_covtype
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score

# Make plots look clean and readable
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
print("Libraries loaded successfully!")

Libraries loaded successfully!


In [2]:
df_1lakh =  pd.read_csv("real_dataset_1lakh.csv")

In [4]:
df_1lakh.head()

,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,...,Soil_Type_30,Soil_Type_31,Soil_Type_32,Soil_Type_33,Soil_Type_34,Soil_Type_35,Soil_Type_36,Soil_Type_37,Soil_Type_38,Soil_Type_39
0,3351.0,206.0,27.0,726.0,124.0,3813.0,192.0,252.0,180.0,2271.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,2732.0,129.0,7.0,212.0,1.0,1082.0,231.0,236.0,137.0,912.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2572.0,24.0,9.0,201.0,25.0,957.0,216.0,222.0,142.0,2191.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2824.0,69.0,13.0,417.0,39.0,3223.0,233.0,214.0,110.0,6478.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2529.0,84.0,5.0,120.0,9.0,1092.0,227.0,231.0,139.0,4983.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
# Select the 10 continuous physical measurement columns
continuous_cols = [
    "Elevation", "Aspect", "Slope",
    "Horizontal_Distance_To_Hydrology", "Vertical_Distance_To_Hydrology",
    "Horizontal_Distance_To_Roadways",
    "Hillshade_9am", "Hillshade_Noon", "Hillshade_3pm",
    "Horizontal_Distance_To_Fire_Points"
]

X = df_1lakh[continuous_cols].copy()

print("Missing values per column:")
print(X.isnull().sum())

print("\nSummary statistics BEFORE scaling:")
display(X.describe().round(2))

Missing values per column:
Elevation                             0
Aspect                                0
Slope                                 0
Horizontal_Distance_To_Hydrology      0
Vertical_Distance_To_Hydrology        0
Horizontal_Distance_To_Roadways       0
Hillshade_9am                         0
Hillshade_Noon                        0
Hillshade_3pm                         0
Horizontal_Distance_To_Fire_Points    0
dtype: int64

Summary statistics BEFORE scaling:


,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points
count,100000.00,100000.00,100000.00,100000.00,100000.00,100000.00,100000.00,100000.00,100000.00,100000.00
mean,2961.09,156.03,14.14,270.19,46.58,2353.24,212.07,223.34,142.61,1982.96
std,279.13,111.98,7.51,213.41,58.49,1559.14,26.86,19.78,38.47,1321.44
min,1871.00,0.00,0.00,0.00,-173.00,0.00,0.00,0.00,0.00,0.00
25%,2811.00,59.00,9.00,108.00,7.00,1110.00,198.00,213.00,119.00,1025.00
50%,2997.00,128.00,13.00,218.00,30.00,2007.00,218.00,226.00,143.00,1717.00
75%,3165.00,261.00,18.00,390.00,69.00,3323.00,231.00,237.00,169.00,2557.00
max,3846.00,360.00,65.00,1383.00,598.00,7087.00,254.00,254.00,253.00,7172.00


In [6]:
# Scale the 1,00,000 rows using StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert back to a DataFrame just to verify Mean ~ 0 and Std ~ 1
X_scaled_df = pd.DataFrame(X_scaled, columns=continuous_cols)
print("Summary statistics AFTER scaling (notice mean ~ 0 and std ~ 1):")
display(X_scaled_df.describe().round(2))

Summary statistics AFTER scaling (notice mean ~ 0 and std ~ 1):


,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points
count,100000.00,100000.00,100000.00,100000.00,100000.00,100000.00,100000.00,100000.00,100000.00,100000.00
mean,0.00,-0.00,0.00,0.00,0.00,0.00,-0.00,-0.00,-0.00,0.00
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
min,-3.91,-1.39,-1.88,-1.27,-3.75,-1.51,-7.90,-11.29,-3.71,-1.50
25%,-0.54,-0.87,-0.68,-0.76,-0.68,-0.80,-0.52,-0.52,-0.61,-0.72
50%,0.13,-0.25,-0.15,-0.24,-0.28,-0.22,0.22,0.13,0.01,-0.20
75%,0.73,0.94,0.51,0.56,0.38,0.62,0.70,0.69,0.69,0.43
max,3.17,1.82,6.78,5.21,9.43,3.04,1.56,1.55,2.87,3.93
